# Step 1: fix p(q) and build the enriched training set

Rescores the saved runs with a cover match, splits out yes/no questions, samples K = 8 closed book answers on 10,000 training questions per dataset for both models, then writes `data/train_enriched.jsonl`.

In [ ]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.


In [1]:
PROJECT = "/content/drive/MyDrive/search_or_think"
import os, sys
from google.colab import drive
drive.mount("/content/drive")
CODE, DATA, OUT = f"{PROJECT}/code", f"{PROJECT}/data", f"{PROJECT}/outputs"
sys.path.insert(0, CODE)

Mounted at /content/drive


## Updated code

In [2]:
%%writefile /content/drive/MyDrive/search_or_think/code/sot/analysis.py
"""Metrics over saved runs: accuracy tables, p(q) distribution, QOR, SOR, and the SAAS vs Soft-SAAS signal check."""
import collections
import json
import math
import random

from .rewards import estimate_boundary, penalty_saas, penalty_soft
from .scoring import has_answer, norm_title


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else math.nan


def by_dataset(rows):
    g = collections.defaultdict(list)
    for r in rows:
        g[r["dataset"]].append(r)
    return dict(sorted(g.items()))


def summarize(rows):
    return {ds: {"n": len(v), "em": mean(r["em"] for r in v), "f1": mean(r["f1"] for r in v),
                 "searches": mean(r["n_search"] for r in v), "format_ok": mean(r["format_ok"] for r in v)}
            for ds, v in by_dataset(rows).items()}


def p_by_id(closed_rows, col="em"):
    """col: "em" (strict) or "cover" (gold answer contained in the prediction)."""
    d = collections.defaultdict(list)
    for r in closed_rows:
        d[r["id"]].append(r[col] >= 1.0)
    return {k: sum(v) / len(v) for k, v in d.items()}


def enriched_sample(recs, pmap, n, mid_frac=1 / 3, seed=0):
    """n questions with about mid_frac at 0 < p(q) < 1 (fewer if the pool has fewer); the rest keep the
    pool's own mix of p(q) = 0 and p(q) = 1."""
    rng = random.Random(seed)
    mid = [r for r in recs if 0 < pmap[r["id"]] < 1]
    rest = [r for r in recs if not 0 < pmap[r["id"]] < 1]
    rng.shuffle(mid)
    rng.shuffle(rest)
    k = min(len(mid), round(n * mid_frac))
    out = mid[:k] + rest[:n - k]
    out += mid[k:k + n - len(out)]  # too few p(q) = 0 or 1 questions: top up with in-between ones
    rng.shuffle(out)
    return out


def p_distribution(pmap, recs):
    out = {}
    for ds, v in by_dataset(recs).items():
        ps = [pmap[r["id"]] for r in v if r["id"] in pmap]
        out[ds] = {"n": len(ps), "hist": dict(sorted(collections.Counter(round(p, 2) for p in ps).items())),
                   "p0": mean(p == 0 for p in ps), "mid": mean(0 < p < 1 for p in ps), "p1": mean(p == 1 for p in ps)}
    return out


def qor(agent_rows, pmap, thr=0.5):
    """Question level over search: share of questions the base model can answer without search
    (p(q) >= thr) on which the policy still searches."""
    out = {}
    for ds, v in by_dataset(agent_rows).items():
        elig = [r for r in v if pmap.get(r["id"], 0.0) >= thr]
        out[ds] = {"eligible": len(elig), "qor": mean(r["n_search"] > 0 for r in elig)}
    return out


def _evidence_complete(rec, titles, text):
    if rec["dataset"] == "hotpotqa":
        gold = {norm_title(t) for t in rec.get("sup_titles", [])}
        return bool(gold) and gold <= titles
    return has_answer(text, rec["golds"])


def sor(agent_rows, recs_by_id):
    """Step level over search: share of searches issued after all gold evidence was already retrieved."""
    out = {}
    for ds, v in by_dataset(agent_rows).items():
        over = total = 0
        for r in v:
            rec, titles, text = recs_by_id[r["id"]], set(), ""
            for s in r["searches"]:
                over += _evidence_complete(rec, titles, text)
                total += 1
                titles |= {norm_title(t) for t in s["titles"]}
                text += " " + " ".join(s["texts"])
        out[ds] = {"searches": total, "sor": over / total if total else math.nan}
    return out


def boundary_signal(closed_rows, sampled_agent_rows):
    """How many questions can each reward penalise at all, and how often correct searching rollouts get penalised."""
    disabled, enabled, ds_of = collections.defaultdict(list), collections.defaultdict(list), {}
    for r in closed_rows:
        disabled[r["id"]].append(r["em"] >= 1.0)
    for r in sampled_agent_rows:
        enabled[r["id"]].append((r["em"] >= 1.0, r["n_search"]))
        ds_of[r["id"]] = r["dataset"]
    out = collections.defaultdict(lambda: {"questions": 0, "labels": collections.Counter(), "signal_saas": 0,
                                           "signal_soft": 0, "correct_searching": 0, "pen_saas": [], "pen_soft": []})
    for qid, rolls in enabled.items():
        if qid not in disabled:
            continue
        b = estimate_boundary(disabled[qid], rolls)
        o = out[ds_of[qid]]
        o["questions"] += 1
        o["labels"][b.label] += 1
        o["signal_saas"] += b.label != "Undetermined"
        o["signal_soft"] += b.p > 0 or b.n_min is not None
        for correct, n in rolls:
            if correct and n > 0:
                o["correct_searching"] += 1
                o["pen_saas"].append(penalty_saas(n, b))
                o["pen_soft"].append(penalty_soft(n, b))
    return {ds: {"questions": o["questions"], "labels": dict(o["labels"]),
                 "signal_saas": o["signal_saas"] / max(o["questions"], 1),
                 "signal_soft": o["signal_soft"] / max(o["questions"], 1),
                 "correct_searching_rollouts": o["correct_searching"],
                 "penalised_saas": mean(p > 0 for p in o["pen_saas"]),
                 "penalised_soft": mean(p > 0 for p in o["pen_soft"]),
                 "mean_pen_saas": mean(o["pen_saas"]), "mean_pen_soft": mean(o["pen_soft"])}
            for ds, o in sorted(out.items())}


Overwriting /content/drive/MyDrive/search_or_think/code/sot/analysis.py


In [3]:
%%writefile /content/drive/MyDrive/search_or_think/code/sot/data.py
"""HotpotQA (distractor) and NQ-open loaders from the Hugging Face Hub, as flat question records."""


def _first_ok(*loaders):
    err = None
    for load in loaders:
        try:
            return load()
        except Exception as e:  # dataset ids move around on the Hub; try the fallback id
            err = e
            print("falling back after:", repr(e)[:150])
    raise err


def load_split(name, split):
    from datasets import load_dataset
    if name == "hotpotqa":
        return _first_ok(lambda: load_dataset("hotpotqa/hotpot_qa", "distractor", split=split),
                         lambda: load_dataset("hotpot_qa", "distractor", split=split, trust_remote_code=True))
    return _first_ok(lambda: load_dataset("google-research-datasets/nq_open", split=split),
                     lambda: load_dataset("nq_open", split=split))


def to_record(name, hf_split, out_split, i, x):
    """Same ids and fields as the original data prep cell, so records from different runs match."""
    if name == "hotpotqa":
        return {"id": x["id"], "dataset": name, "split": out_split, "question": x["question"],
                "golds": [x["answer"]], "sup_titles": sorted(set(x["supporting_facts"]["title"])),
                "type": x["type"], "level": x["level"]}
    return {"id": f"nq_{hf_split}_{i}", "dataset": name, "split": out_split, "question": x["question"],
            "golds": list(x["answer"]), "sup_titles": []}


Overwriting /content/drive/MyDrive/search_or_think/code/sot/data.py


In [4]:
%%writefile /content/drive/MyDrive/search_or_think/code/run_inference.py
"""Inference only experiments for one model (no RL). Writes one jsonl per task to outputs/<model>/.

Tasks
  closed_sampled  K search disabled samples (T=1.0) on probe + test  -> p(q)
  no_search       greedy, no retrieval, test                         -> baseline
  rag             one forced search with the question, greedy, test  -> baseline
  agent           prompted search agent, greedy, test                -> baseline + QOR / SOR
  agent_sampled   K prompted agent samples (T=1.0) on probe          -> N_min, SAAS labels, reward signal check
  closed_pool     K search disabled samples (T=1.0) on the training pool -> p(q) for the enriched training set
"""
import argparse
import json
import os
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from sot.agent import agent_instruction, closed_book_instruction, format_info, rag_instruction, run_agent
from sot.retrieval import Retriever
from sot.scoring import em, f1

ALL_TASKS = "closed_sampled,no_search,rag,agent,agent_sampled"


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def to_row(rec, s, sample):
    return {"id": rec["id"], "dataset": rec["dataset"], "split": rec["split"], "sample": sample,
            "answer": s["answer"], "format_ok": s["format_ok"], "em": em(s["answer"], rec["golds"]),
            "f1": f1(s["answer"], rec["golds"]), "n_search": s["n_search"], "stop_reason": s["stop_reason"],
            "searches": [{"query": x["query"], "titles": [p["title"] for p in x["passages"]],
                          "texts": [p["text"] for p in x["passages"]]} for x in s["searches"]],
            "text": s["text"]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--project", required=True)
    ap.add_argument("--n", type=int, default=500, help="questions per dataset per split")
    ap.add_argument("--k", type=int, default=4, help="samples per question for p(q) and N_min")
    ap.add_argument("--max_searches", type=int, default=4)
    ap.add_argument("--tasks", default=ALL_TASKS)
    ap.add_argument("--pool_n", type=int, default=10000, help="pool questions per dataset for closed_pool")
    ap.add_argument("--tag", default="")
    ap.add_argument("--max_model_len", type=int, default=8192)
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    args = ap.parse_args()

    from vllm import LLM, SamplingParams

    data = os.path.join(args.project, "data")
    test = sum((load_jsonl(f"{data}/{d}_test.jsonl")[:args.n] for d in ("hotpotqa", "nq")), [])
    probe = sum((load_jsonl(f"{data}/{d}_probe.jsonl")[:args.n] for d in ("hotpotqa", "nq")), [])
    tasks = args.tasks.split(",")
    pool = sum((load_jsonl(f"{data}/{d}_pool.jsonl")[:args.pool_n] for d in ("hotpotqa", "nq")), []) \
        if "closed_pool" in tasks else []
    name = args.model.split("/")[-1] + (f"_{args.tag}" if args.tag else "")
    out_dir = os.path.join(args.project, "outputs", name)
    os.makedirs(out_dir, exist_ok=True)

    llm = LLM(model=args.model, dtype="bfloat16", gpu_memory_utilization=args.gpu_mem,
              max_model_len=args.max_model_len, seed=0, enable_prefix_caching=True)
    tok = llm.get_tokenizer()
    retrieve = Retriever(k=3)

    def chat(instr):
        return tok.apply_chat_template([{"role": "user", "content": instr}], tokenize=False, add_generation_prompt=True)

    def generate(prompts, stop, max_tokens, temperature):
        lens = [len(x) for x in tok(prompts, add_special_tokens=False)["input_ids"]]
        keep = [i for i, n in enumerate(lens) if n + max_tokens <= args.max_model_len]  # too long -> "" -> agent stops it
        outs = [""] * len(prompts)
        if keep:
            sp = SamplingParams(temperature=temperature, top_p=1.0, max_tokens=max_tokens, stop=stop,
                                include_stop_str_in_output=True)
            for i, r in zip(keep, llm.generate([prompts[i] for i in keep], sp, use_tqdm=True)):
                outs[i] = r.outputs[0].text
        return outs

    def run(task, recs, instr_fn, k=1, temperature=0.0, allow_search=True, max_turn_tokens=512, rag=False,
            prefix=""):
        """prefix starts the assistant turn ("<answer>" or "<think>") so small models stay in the tag format."""
        t0 = time.time()
        recs_k = [r for r in recs for _ in range(k)]
        prefill = [{"text": prefix, "searches": []} for _ in recs_k]
        if rag:
            passages = retrieve([r["question"] for r in recs_k])
            prefill = [{"text": f"<search> {r['question']} </search>" + format_info(p) + prefix,
                        "searches": [{"query": r["question"], "passages": p}]} for r, p in zip(recs_k, passages)]
        states = run_agent(generate, retrieve, [chat(instr_fn(r["question"])) for r in recs_k],
                           allow_search=allow_search, max_searches=args.max_searches,
                           max_turn_tokens=max_turn_tokens, temperature=temperature, prefill=prefill)
        rows = [to_row(r, s, i % k) for i, (r, s) in enumerate(zip(recs_k, states))]
        with open(f"{out_dir}/{task}.jsonl", "w") as f:
            f.writelines(json.dumps(x) + "\n" for x in rows)
        mins = (time.time() - t0) / 60
        print(f"[{name}] {task}: {len(rows)} trajectories in {mins:.1f} min, "
              f"EM={sum(x['em'] for x in rows) / len(rows):.3f}, "
              f"searches={sum(x['n_search'] for x in rows) / len(rows):.2f}", flush=True)
        return mins

    agent_instr = lambda q: agent_instruction(q, args.max_searches)
    jobs = {
        "closed_sampled": lambda: run("closed_sampled", probe + test, closed_book_instruction, k=args.k,
                                      temperature=1.0, allow_search=False, max_turn_tokens=32, prefix="<answer>"),
        "no_search": lambda: run("no_search", test, closed_book_instruction, allow_search=False, max_turn_tokens=32,
                                 prefix="<answer>"),
        "rag": lambda: run("rag", test, rag_instruction, allow_search=False, max_turn_tokens=32, rag=True,
                           prefix="<answer>"),
        "agent": lambda: run("agent", test, agent_instr, prefix="<think>"),
        "agent_sampled": lambda: run("agent_sampled", probe, agent_instr, k=args.k, temperature=1.0, prefix="<think>"),
        "closed_pool": lambda: run("closed_pool", pool, closed_book_instruction, k=args.k, temperature=1.0,
                                   allow_search=False, max_turn_tokens=32, prefix="<answer>"),
    }
    timings = {t: jobs[t]() for t in tasks}
    tpath = f"{out_dir}/timings.json"  # merged, so a later partial run keeps the earlier tasks' entries
    log = json.load(open(tpath)) if os.path.exists(tpath) else {"model": args.model, "n": args.n, "minutes": {}}
    log["minutes"].update(timings)
    log.setdefault("k_by_task", {}).update({t: args.k for t in timings})
    with open(tpath, "w") as f:
        json.dump(log, f, indent=2)


if __name__ == "__main__":
    main()


Overwriting /content/drive/MyDrive/search_or_think/code/run_inference.py


In [5]:
%%writefile /content/drive/MyDrive/search_or_think/code/test_sot.py
"""Unit tests: python -m pytest -q test_sot.py"""
import pytest

from sot.agent import extract_answer, run_agent
from sot.analysis import boundary_signal, enriched_sample, p_by_id, qor, sor
from sot.rewards import (Boundary, WarmupTrigger, estimate_boundary, reward_fixed, reward_plain, reward_saas,
                         reward_soft_saas)
from sot.scoring import em, f1, has_answer

A = 0.1  # alpha


# scoring
def test_scoring():
    assert em("The Eiffel Tower!", ["eiffel tower"]) == 1.0
    assert f1("Paris France", ["Paris"]) == pytest.approx(2 / 3)
    assert f1("yes", ["no"]) == 0.0 and f1("no", ["no"]) == 1.0
    assert has_answer("Built in 1889 by Eiffel.", ["1889"]) and not has_answer("in 18890", ["1889"])


# rewards
def test_plain_and_fixed():
    assert reward_plain(0.7, 3) == 0.7
    assert reward_fixed(1.0, 2, 0.1) == pytest.approx(0.8)


def test_wrong_answers_never_penalised():
    for b in [Boundary(1.0, 0, "NoSearch"), Boundary(0.0, 1, "NeedSearch"), Boundary(0.5, 1, "Undetermined")]:
        assert reward_saas(0.5, 4, b, A) == 0.5
        assert reward_soft_saas(0.5, 4, b, A) == 0.5


def test_saas():
    assert reward_saas(1.0, 2, Boundary(1.0, 0, "NoSearch"), A) == pytest.approx(0.8)
    assert reward_saas(1.0, 3, Boundary(0.0, 1, "NeedSearch"), A) == pytest.approx(0.8)
    assert reward_saas(1.0, 1, Boundary(0.0, 1, "NeedSearch"), A) == 1.0
    assert reward_saas(1.0, 4, Boundary(0.5, 1, "Undetermined"), A) == 1.0  # no signal


def test_soft_saas_matches_saas_at_extremes_and_mixes_between():
    assert reward_soft_saas(1.0, 2, Boundary(1.0, 0, "NoSearch"), A) == pytest.approx(0.8)
    assert reward_soft_saas(1.0, 3, Boundary(0.0, 1, "NeedSearch"), A) == pytest.approx(0.8)
    # p = 0.5, N = 3, N_min = 1: 0.5*3 + 0.5*2 = 2.5 searches of penalty
    assert reward_soft_saas(1.0, 3, Boundary(0.5, 1, "Undetermined"), A) == pytest.approx(0.75)
    # never answerable and no correct searching rollout: only searches beyond the cap count
    assert reward_soft_saas(1.0, 4, Boundary(0.0, None, "Undetermined"), A) == 1.0
    assert reward_soft_saas(1.0, 2, Boundary(1.0, 0, "NoSearch"), A, w=0.0) == 1.0  # warm up off


def test_estimate_boundary():
    assert estimate_boundary([True] * 4, [(True, 1)]).label == "NoSearch"
    b = estimate_boundary([False] * 4, [(False, 4), (True, 2), (True, 1)])
    assert (b.label, b.n_min, b.p) == ("NeedSearch", 1, 0.0)
    b = estimate_boundary([True, False, False, False], [(True, 2)])
    assert (b.label, b.p) == ("Undetermined", 0.25)
    assert estimate_boundary([False] * 4, [(False, 3)]).label == "Undetermined"


def test_warmup_trigger():
    w = WarmupTrigger(window=5, eps=0.01, ramp=10, ma=3)
    ws = [w.update(min(0.05 * t, 0.6)) for t in range(60)]  # rises for 12 steps, then flat
    assert w.trigger_step is not None and 12 < w.trigger_step < 30
    assert all(x == 0.0 for x in ws[:w.trigger_step + 1])
    assert ws[-1] == 1.0 and ws == sorted(ws)


# agent loop with a fake model and retriever
def fake_retrieve(queries):
    return [[{"title": f"T_{q}", "text": f"text about {q}"}] for q in queries]


def make_generate(script):
    """script: prompt keyword -> list of outputs, one per turn."""
    calls = {}

    def gen(prompts, stop, max_tokens, temperature):
        outs = []
        for p in prompts:
            key = next(k for k in script if k in p)
            turn = calls.get(p.split("|")[0], 0)
            calls[p.split("|")[0]] = turn + 1
            outs.append(script[key][turn] if turn < len(script[key]) else "")
        return outs
    return gen


def test_run_agent_paths():
    script = {
        "Q_direct": ["<think>know it</think><answer> Paris </answer>"],
        "Q_search": ["<think>?</think><search> eiffel </search>", "<think>ok</think><answer>1889</answer>"],
        "Q_greedy": ["<search> a </search>"] * 6 + ["<answer>x</answer>"],
        "Q_rambles": ["no tags here just text\nfinal guess Rome"],
    }
    prompts = [f"{k}|" for k in script]
    st = run_agent(make_generate(script), fake_retrieve, prompts, max_searches=2)
    d, s, g, r = st
    assert (d["answer"], d["n_search"], d["stop_reason"]) == ("Paris", 0, "answer")
    assert (s["answer"], s["n_search"]) == ("1889", 1) and "<info>" in s["text"] and "T_eiffel" in s["text"]
    assert g["n_search"] == 2 and g["stop_reason"] == "search_limit" and "Search limit reached" in g["text"]
    assert r["format_ok"] is False and r["answer"] == "final guess Rome" and r["stop_reason"] == "no_tag"


def test_closed_book_and_prefill():
    gen = make_generate({"Q": ["<think>x</think><answer>a</answer>"]})
    st = run_agent(gen, None, ["Q|"], allow_search=False,
                   prefill=[{"text": "<search> Q </search>", "searches": [{"query": "Q", "passages": []}]}])
    assert st[0]["answer"] == "a" and st[0]["n_search"] == 1
    assert extract_answer("<answer>a</answer> then <answer> b </answer>") == "b"
    assert extract_answer("<answer><answer> Paris </answer>") == "Paris"  # model repeats the prefilled tag
    assert extract_answer("<answer> Paris, France\nmore text", lenient=True) == "Paris, France"  # never closed
    assert extract_answer("<answer> Paris, France\nmore text") is None


# metrics
def test_metrics():
    recs = {"h1": {"dataset": "hotpotqa", "sup_titles": ["A", "B"], "golds": ["x"]},
            "n1": {"dataset": "nq", "sup_titles": [], "golds": ["1889"]}}
    agent = [
        {"id": "h1", "dataset": "hotpotqa", "n_search": 3, "searches": [
            {"titles": ["A"], "texts": [""]}, {"titles": ['"B"'], "texts": [""]}, {"titles": ["C"], "texts": [""]}]},
        {"id": "n1", "dataset": "nq", "n_search": 2, "searches": [
            {"titles": ["E"], "texts": ["built in 1889"]}, {"titles": ["E"], "texts": [""]}]},
    ]
    s = sor(agent, recs)
    assert s["hotpotqa"]["sor"] == pytest.approx(1 / 3) and s["nq"]["sor"] == pytest.approx(1 / 2)
    closed = [{"id": "h1", "dataset": "hotpotqa", "em": e} for e in (1, 1, 0, 0)] + \
             [{"id": "n1", "dataset": "nq", "em": 0}] * 4
    pm = p_by_id(closed)
    assert pm == {"h1": 0.5, "n1": 0.0}
    assert qor(agent, pm)["hotpotqa"]["qor"] == 1.0
    sampled = [{"id": "h1", "dataset": "hotpotqa", "em": 1, "n_search": 2},
               {"id": "n1", "dataset": "nq", "em": 1, "n_search": 3},
               {"id": "n1", "dataset": "nq", "em": 1, "n_search": 1}]
    assert p_by_id([{"id": "a", "em": 0.0, "cover": 1.0}, {"id": "a", "em": 0.0, "cover": 0.0}], "cover") == {"a": 0.5}
    sig = boundary_signal(closed, sampled)
    assert sig["hotpotqa"]["signal_saas"] == 0 and sig["hotpotqa"]["signal_soft"] == 1  # p = 0.5 only Soft sees it
    assert sig["nq"]["labels"] == {"NeedSearch": 1} and sig["nq"]["penalised_saas"] == 0.5


def test_enriched_sample():
    recs = [{"id": str(i)} for i in range(100)]
    pmap = {str(i): 0.5 if i < 10 else 0.0 if i < 90 else 1.0 for i in range(100)}
    out = enriched_sample(recs, pmap, 30)
    assert len(out) == 30 and sum(0 < pmap[r["id"]] < 1 for r in out) == 10
    assert len({r["id"] for r in out}) == 30
    out = enriched_sample(recs, pmap, 60)  # only 10 in-between questions exist: take all, fill with the rest
    assert len(out) == 60 and sum(0 < pmap[r["id"]] < 1 for r in out) == 10
    flip = {k: 0.5 if v == 0.0 else v for k, v in pmap.items()}  # 90 in between, 10 at p = 1
    out = enriched_sample(recs, flip, 60)
    assert len(out) == 60 and sum(flip[r["id"]] == 1.0 for r in out) == 10


Overwriting /content/drive/MyDrive/search_or_think/code/test_sot.py


In [6]:
!cd {CODE} && python -m pytest -q test_sot.py

...........                                                              [100%]
11 passed in 8.39s


## Training pool: 10,000 questions per dataset, disjoint from the probe

In [ ]:
import json
import random

from sot.analysis import load_jsonl
from sot.data import load_split, to_record

POOL_N = 10000  # training questions per dataset to screen for p(q)
for name in ("hotpotqa", "nq"):
    taken = {r["id"] for r in load_jsonl(f"{DATA}/{name}_probe.jsonl")}  # keep the pool disjoint from the probe
    d = load_split(name, "train")
    rows = []
    for i in random.Random(7).sample(range(len(d)), min(len(d), POOL_N + len(taken))):
        r = to_record(name, "train", "pool", i, d[i])
        if r["id"] not in taken:
            rows.append(r)
        if len(rows) == POOL_N:
            break
    with open(f"{DATA}/{name}_pool.jsonl", "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)
    print(f"{name}_pool: {len(rows)} questions")


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

... (8 outputs left out)

nq_open/validation-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87925 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3610 [00:00<?, ? examples/s]

nq_pool: 10000 questions


## K = 8 closed book samples on the pool

In [ ]:
!cd {CODE} && python run_inference.py --model Qwen/Qwen2.5-0.5B-Instruct --project {PROJECT} --tasks closed_pool --k 8

(MainProcess pid=4037) INFO 10-25 17:43:05 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
config.json: 100% 659/659 [00:00<00:00, 2.97MB/s]
(MainProcess pid=4037) INFO 10-25 17:43:23 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=4037) INFO 10-25 17:43:23 [model.py:2152] Using max model len 8192
(MainProcess pid=4037) INFO 10-25 17:43:23 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=4037) INFO 10-25 17:43:24 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0

In [ ]:
!cd {CODE} && python run_inference.py --model Qwen/Qwen2.5-1.5B-Instruct --project {PROJECT} --tasks closed_pool --k 8

(MainProcess pid=6247) INFO 10-25 17:50:17 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
config.json: 100% 660/660 [00:00<00:00, 3.06MB/s]
(MainProcess pid=6247) INFO 10-25 17:50:19 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=6247) INFO 10-25 17:50:19 [model.py:2152] Using max model len 8192
(MainProcess pid=6247) INFO 10-25 17:50:19 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=6247) INFO 10-25 17:50:20 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0

## Results and the enriched training set


In [7]:
import os, shutil
for f in (f"{DATA}/train_enriched.jsonl", f"{OUT}/pq_fix_summary.json"):
    if os.path.exists(f):
        shutil.copy(f, "/content/" + os.path.basename(f) + ".before")
        print("backed up", f)

backed up /content/drive/MyDrive/search_or_think/data/train_enriched.jsonl
backed up /content/drive/MyDrive/search_or_think/outputs/pq_fix_summary.json


In [8]:
import json
import os

import pandas as pd

from sot.analysis import enriched_sample, load_jsonl, mean, p_by_id
from sot.scoring import has_answer, normalize_answer

MODELS = ["Qwen2.5-0.5B-Instruct", "Qwen2.5-1.5B-Instruct"]
TRAIN_MODEL = "Qwen2.5-0.5B-Instruct"  # model whose p(q) picks the training set
P_METRIC = "cover"                     # cover match for the knowledge estimate; rewards still use F1
N_TRAIN_PER_DATASET = 1500             # few thousand questions in total, as in the proposal

recs = {r["id"]: r for ds in ("hotpotqa", "nq") for sp in ("test", "probe", "pool")
        if os.path.exists(f"{DATA}/{ds}_{sp}.jsonl") for r in load_jsonl(f"{DATA}/{ds}_{sp}.jsonl")}


def yesno(r):
    return normalize_answer(r["golds"][0]) in ("yes", "no")


def dist(pmap, subset):
    ps = [pmap[r["id"]] for r in subset if r["id"] in pmap]
    return {"n": len(ps), "p0": mean(p == 0 for p in ps), "mid": mean(0 < p < 1 for p in ps),
            "p1": mean(p == 1 for p in ps)}


table, pmaps = [], {}
for m in MODELS:
    for task, split, k in [("closed_sampled", "probe", 4), ("closed_pool", "pool", 8)]:
        path = f"{OUT}/{m}/{task}.jsonl"
        if not os.path.exists(path):
            print("missing", path)
            continue
        rows = load_jsonl(path)
        for r in rows:
            r["cover"] = float(has_answer(r["answer"] or "", recs[r["id"]]["golds"]))
        for metric in ("em", "cover"):
            pmap = pmaps[(m, task, metric)] = p_by_id(rows, metric)
            for ds in ("hotpotqa", "nq"):
                sub = [r for r in recs.values() if r["dataset"] == ds and r["split"] == split]
                groups = [("all", sub)] + ([("no yes/no", [r for r in sub if not yesno(r)])] if ds == "hotpotqa" else [])
                for g, s in groups:
                    table.append({"model": m.split("-")[1], "K": k, "metric": metric, "dataset": ds, "questions": g,
                                  **dist(pmap, s)})
df = pd.DataFrame(table)
print(df.round(3).to_string(index=False))

# Boundary-enriched training set from the pool, using the training model's p(q)
pmap = pmaps[(TRAIN_MODEL, "closed_pool", P_METRIC)]
train, built = [], {}
for ds in ("hotpotqa", "nq"):
    pool = [r for r in recs.values() if r["dataset"] == ds and r["split"] == "pool"]
    picked = enriched_sample(pool, pmap, N_TRAIN_PER_DATASET, seed=0)
    train += [{**r, "split": "train", "p_pool": pmap[r["id"]]} for r in picked]
    built[ds] = {"pool": len(pool), "pool_mid": dist(pmap, pool)["mid"], "picked": len(picked),
                 "picked_mid": dist(pmap, picked)["mid"]}
with open(f"{DATA}/train_enriched.jsonl", "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in train)
print(f"\nwrote {DATA}/train_enriched.jsonl ({len(train)} questions, p(q) from {TRAIN_MODEL}, {P_METRIC} match)")

summary = {"table": df.round(4).to_dict(orient="records"), "enriched": built}
with open(f"{OUT}/pq_fix_summary.json", "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(built, indent=1))
print(df.round(3).to_csv(index=False))


model  K metric  dataset questions     n    p0   mid    p1
 0.5B  4     em hotpotqa       all   500 0.868 0.106 0.026
 0.5B  4     em hotpotqa no yes/no   464 0.892 0.080 0.028
 0.5B  4     em       nq       all   500 0.968 0.024 0.008
 0.5B  4  cover hotpotqa       all   500 0.798 0.128 0.074
 0.5B  4  cover hotpotqa no yes/no   464 0.845 0.112 0.043
 0.5B  4  cover       nq       all   500 0.928 0.058 0.014
 0.5B  8     em hotpotqa       all 10000 0.851 0.138 0.012
 0.5B  8     em hotpotqa no yes/no  9409 0.879 0.108 0.012
 0.5B  8     em       nq       all 10000 0.941 0.059 0.001
 0.5B  8  cover hotpotqa       all 10000 0.807 0.157 0.036
 0.5B  8  cover hotpotqa no yes/no  9409 0.842 0.139 0.019
 0.5B  8  cover       nq       all 10000 0.882 0.114 0.004
 1.5B  4     em hotpotqa       all   500 0.772 0.142 0.086
 1.5B  4     em hotpotqa no yes/no   464 0.819 0.142 0.039
 1.5B  4     em       nq       all   500 0.886 0.094 0.020
 1.5B  4  cover hotpotqa       all   500 0.740 0.166 0.0

In [9]:
import filecmp
same = filecmp.cmp(f"{DATA}/train_enriched.jsonl", "/content/train_enriched.jsonl.before", shallow=False)
print("training set identical" if same else "TRAINING SET CHANGED: restore it now")
# restore if changed:
# shutil.copy("/content/train_enriched.jsonl.before", f"{DATA}/train_enriched.jsonl")

training set identical


In [ ]:
from google.colab import runtime
runtime.unassign()